# Lens extraction — vocabulary-space readout (docs/specs/lens_spec.md)

GPU stages of the lens study, one causal decoder per run: (1) dump the
per-layer hidden states at the **generating position** for every board, and
(2) train **tuned-lens translators** on generic text. Everything downstream
(`lens-apply`, `lens-analyze`) runs locally on the dump — no GPU needed.

Set `MODEL_KEY` in the config cell and run top-to-bottom once per model
(`mistral`, `qwen`, `qwen_random`). Resume is ON: if the runtime dies,
re-run the extraction cell and it continues from the last checkpoint.

## Setup (run cells 1–3 once per session)

In [ ]:
# Cell 1 — Clone or update the package code from GitHub.
import os
REPO_URL = "https://github.com/JoaoPedroFPK/codenames-interpretability.git"
REPO_DIR = "/content/codenames-interpretability"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

In [ ]:
# Cell 2 — Install the package in editable mode, WITH the [lens] extra
# (adds `datasets` for the tuned-lens training text; core pins unchanged).
!pip install -q -e "{REPO_DIR}[lens]" 

In [ ]:
# === Config ===
# MODEL_KEY: which causal decoder this session runs. One of:
#   "mistral"      -> Mistral-7B-Instruct-v0.2   (prefix: mistral)
#   "qwen"         -> Qwen2.5-7B-Instruct        (prefix: qwen)
#   "qwen_random"  -> random-init Qwen null      (prefix: random_qwen)
MODEL_KEY = "mistral"

# SAMPLE_SIZE: None -> FULL dataset (the lens study is pre-registered on the
# full corpus, docs/specs/lens_spec.md §4); <int> -> quick smoke run.
SAMPLE_SIZE = None

# Conditions to extract. no_social is the primary pre-registered condition;
# add "with_social" for the robustness pass (doubles GPU time and storage).
CONDITIONS = ("no_social",)

USE_FLASH_ATTN = True      # trained models only; ignored for qwen_random
TRAIN_TUNED    = True      # run the tuned-lens training stage (Cell 7)
TUNE_STEPS     = 250
TUNE_SEQ_LEN   = 512
TUNE_MAX_CHARS = 20_000_000

print(f"Lens run: {MODEL_KEY}, "
      + ("FULL dataset" if SAMPLE_SIZE is None else f"{SAMPLE_SIZE} boards")
      + f", conditions={CONDITIONS}")

In [ ]:
# Cell 2b — Verify the installed environment matches the pinned set.
!codenames-experiment doctor --model {MODEL_KEY}

In [ ]:
# Cell 3 — Autoreload (so package edits flow through without restarting) and mount Drive.
import sys, types, importlib
if "imp" not in sys.modules:
    # Python 3.12 removed `imp`; old IPython autoreload still imports it.
    _imp = types.ModuleType("imp")
    _imp.reload = importlib.reload
    sys.modules["imp"] = _imp

# Put the freshly cloned package on sys.path of THIS kernel (see 01-07 for why).
REPO_DIR = "/content/codenames-interpretability"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount("/content/drive")

## Stage 1 — Hidden-state extraction at the generating position

In [ ]:
# Cell 4 — Imports, loader dispatch, load model.
import dataclasses
import importlib
from codenames.contract import CONTRACT_V1
from codenames.data import load_dataset, sample_turns
from codenames.lens.extract import run_lens_extraction

_LOADERS = {
    "mistral":     ("codenames.models.mistral",     "load_mistral_instruct"),
    "qwen":        ("codenames.models.qwen",        "load_qwen_instruct"),
    "qwen_random": ("codenames.models.qwen_random", "load_qwen_random"),
}
_module, _attr = _LOADERS[MODEL_KEY]
loader = getattr(importlib.import_module(_module), _attr)

if MODEL_KEY in ("mistral", "qwen") and USE_FLASH_ATTN:
    model, tokenizer, meta = loader(attn_implementation="flash_attention_2")
else:
    model, tokenizer, meta = loader()
print(f"Model loaded: {meta['model_name']} (prefix: {meta['prefix']})")
print(f"  Layers: {meta['num_layers']}, Hidden dim: {meta['hidden_dim']}")

In [ ]:
# Cell 5 — Load and sample dataset (same seeded draw as the thesis runs).
DATASET_PATH = "/content/drive/MyDrive/TCC/clue_generation.csv"

df = load_dataset(DATASET_PATH)
n_boards = len(df) if SAMPLE_SIZE is None else SAMPLE_SIZE
CONTRACT = dataclasses.replace(CONTRACT_V1, sample_size=n_boards)
df_sample = sample_turns(df, n=CONTRACT.sample_size, seed=CONTRACT.random_seed)
print(f"Sample size: {len(df_sample)} boards")
print(f"First 10 row_ids: {sorted(df_sample['row_id'].tolist())[:10]}")

In [ ]:
# Cell 6 — Run lens extraction (resumable).
# Writes into the SAME Drive output dir as the thesis run for this model:
#   {prefix}_lens_hidden_{mode}_f16.npy   fp16 memmap [N, layers+1, d] (~2.1 GB full)
#   {prefix}_lens_index_{mode}.csv        board_idx, row_id, token count, ok
#   {prefix}_lens_readout_f16.npz         final RMSNorm + lm_head (once per model)
BASE_DIR = f"/content/drive/MyDrive/TCC/{meta['prefix']}_outputs"
CHECKPOINT_DIR = f"/content/drive/MyDrive/TCC/{meta['prefix']}_checkpoints"

# Resume — ENABLED. The manifest commits every ~200 boards; if the runtime
# dies, re-run this cell and it continues (completed conditions are skipped).
RESUME = True

paths = run_lens_extraction(
    model=model,
    tokenizer=tokenizer,
    df=df_sample,
    base_dir=BASE_DIR,
    prefix=meta["prefix"],
    contract=CONTRACT,
    chat_template_strategy=meta["chat_template_strategy"],
    num_layers=meta["num_layers"],
    hidden_dim=meta["hidden_dim"],
    conditions=CONDITIONS,
    device=meta["device"],
    resume=RESUME,
    checkpoint_dir=CHECKPOINT_DIR,
)
paths

## Stage 2 — Tuned-lens translator training (generic text, no labels)

In [ ]:
# Cell 7 — Train tuned-lens translators and save them next to the dump.
# Trained to match the model's OWN final logits on WikiText-103 — never the
# human targets (docs/specs/lens_spec.md §6 no-leakage argument). ~1-2 h on an A100.
if TRAIN_TUNED:
    from datasets import load_dataset as hf_load_dataset
    from codenames.lens.tuned import TunedLensConfig, train_tuned_lens

    ds = hf_load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
    texts, total = [], 0
    for rec in ds:
        t = rec["text"]
        if t.strip():
            texts.append(t)
            total += len(t)
        if total >= TUNE_MAX_CHARS:
            break
    print(f"Training corpus: {len(texts)} passages, {total/1e6:.1f}M chars")

    cfg = TunedLensConfig(seq_len=TUNE_SEQ_LEN, n_steps=TUNE_STEPS)
    lens = train_tuned_lens(model, tokenizer, texts, cfg)
    out = f"{BASE_DIR}/{meta['prefix']}_lens_translators.npz"
    lens.save(out)
    print(f"Saved: {out} (final loss {lens.history[-1]:.4f})")
else:
    print("TRAIN_TUNED = False — skipped.")

## Next steps (local, no GPU)

Download the new `*_lens_*` files from Drive into `output/{prefix}_outputs/`,
then on your machine:

```bash
codenames-experiment lens-apply   --model mistral --dataset data/clue_generation.csv \
    --output-dir output/mistral_outputs --full --conditions no_social
codenames-experiment lens-analyze --output-root output \
    --models mistral,qwen --random-model random_qwen --condition no_social
```

`lens-analyze` applies the pre-registered docs/specs/lens_spec.md §3 rules. **Read the
calibration gate first** (`output/lens_analysis/lens_calibration_no_social.csv`):
if final-layer raw-lens agreement with generation is below 0.95 for a model,
that model's results are BLOCKED pending a pipeline-bug hunt — do not
interpret its curves. Then check the random_qwen null is flat before reading
any trajectory classification.